# FIFA World Cup Match Prediction Engine

This notebook prepares the prediction pipeline used by the web application.

The objective is to generate the latest statistics for each national team, construct feature vectors for unseen matches, and use the trained XGBoost model to predict Home Win, Draw, and Away Win probabilities.

In [1]:

import pandas as pd
import numpy as np
import joblib

In [2]:
matches = pd.read_csv("../datasets/results.csv")

fifa_rankings = pd.read_csv("../datasets/fifa_rankings_.csv")

elo_ratings = pd.read_csv("../datasets/elo_ratings_wc2026.csv")

fixtures = pd.read_csv("../datasets/wc_2026_fixtures.csv")

teams = pd.read_csv("../datasets/wc_2026_teams.csv")

In [3]:
model = joblib.load("../models/fifa_worldcup_xgboost.pkl")

feature_columns = joblib.load("../models/feature_columns.pkl")

In [4]:
matches["date"] = pd.to_datetime(matches["date"])

recent_matches = matches[
    matches["date"] >= "2006-01-01"
].copy()

print(recent_matches.shape)

(19745, 9)


In [5]:
completed_matches = recent_matches.dropna(
    subset=["home_score", "away_score"]
).copy()

print(completed_matches.shape)

(19701, 9)


In [6]:
print(teams.columns)

Index(['team', 'group', 'confederation', 'fifa_rank', 'coach',
       'best_wc_result', 'debut_2026'],
      dtype='object')


In [7]:
teams_list = sorted(
    teams["team"].tolist()
)

print(len(teams_list))

48


## Retrieving Recent Matches

A helper function is created to retrieve the five most recent completed matches played by a national team.

In [8]:
TEAM_NAME_MAP = {
    "USA": "United States",
    "Türkiye": "Turkey",
    "Czechia": "Czech Republic"
}

In [9]:
def last_five_matches(team):

    search_team = TEAM_NAME_MAP.get(team, team)

    home = completed_matches[
        completed_matches["home_team"] == search_team
    ]

    away = completed_matches[
        completed_matches["away_team"] == search_team
    ]

    team_matches = pd.concat([home, away])

    team_matches = team_matches.sort_values("date")

    return team_matches.tail(5)

In [10]:
last_five_matches("Brazil")

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49122,2026-03-26,Brazil,France,1.0,2.0,Friendly,Foxborough,United States,True
49220,2026-03-31,Brazil,Croatia,3.0,1.0,Friendly,Orlando,United States,True
49276,2026-05-31,Brazil,Panama,6.0,2.0,Friendly,Rio de Janeiro,Brazil,False
49347,2026-06-06,Brazil,Egypt,2.0,1.0,Friendly,Cleveland,United States,True
49410,2026-06-13,Brazil,Morocco,1.0,1.0,FIFA World Cup,East Rutherford,United States,True


In [11]:
last_five_matches("USA")

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49186,2026-03-28,United States,Belgium,2.0,5.0,Friendly,Atlanta,United States,False
49248,2026-03-31,United States,Portugal,0.0,2.0,Friendly,Atlanta,United States,False
49283,2026-05-31,United States,Senegal,3.0,2.0,Friendly,Charlotte,United States,False
49359,2026-06-06,United States,Germany,1.0,2.0,Friendly,Chicago,United States,False
49408,2026-06-12,United States,Paraguay,4.0,1.0,FIFA World Cup,Inglewood,United States,False


In [12]:
last_five_matches("Türkiye")

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49135,2026-03-26,Turkey,Romania,1.0,0.0,FIFA World Cup qualification,Istanbul,Turkey,False
49254,2026-03-31,Kosovo,Turkey,0.0,1.0,FIFA World Cup qualification,Pristina,Kosovo,False
49291,2026-06-01,Turkey,North Macedonia,4.0,0.0,Friendly,Istanbul,Turkey,False
49362,2026-06-06,Venezuela,Turkey,1.0,2.0,Friendly,Fort Lauderdale,United States,True
49412,2026-06-13,Australia,Turkey,2.0,0.0,FIFA World Cup,Vancouver,Canada,True


In [13]:
last_five_matches("Czechia")

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49255,2026-03-31,Czech Republic,Denmark,2.0,2.0,FIFA World Cup qualification,Prague,Czech Republic,False
49278,2026-05-31,Czech Republic,Kosovo,2.0,1.0,Friendly,Prague,Czech Republic,False
49311,2026-06-04,Guatemala,Czech Republic,1.0,5.0,Friendly,Harrison,United States,True
49406,2026-06-11,South Korea,Czech Republic,2.0,1.0,FIFA World Cup,Zapopan,Mexico,True
49429,2026-06-18,Czech Republic,South Africa,1.0,1.0,FIFA World Cup,Atlanta,United States,True


In [14]:
def compute_team_statistics(team):

    last5 = last_five_matches(team)

    if len(last5) == 0:
        return None

    wins = 0
    draws = 0
    losses = 0

    goals_scored = 0
    goals_conceded = 0

    form_points = 0

    for _, match in last5.iterrows():

        search_team = TEAM_NAME_MAP.get(team, team)

        if match["home_team"] == search_team:

            scored = match["home_score"]
            conceded = match["away_score"]

        else:

            scored = match["away_score"]
            conceded = match["home_score"]

        goals_scored += scored
        goals_conceded += conceded

        if scored > conceded:

            wins += 1
            form_points += 3

        elif scored == conceded:

            draws += 1
            form_points += 1

        else:

            losses += 1

    return {

        "team": team,

        "wins_last5": wins,

        "draws_last5": draws,

        "losses_last5": losses,

        "form_points": form_points,

        "goals_scored": goals_scored,

        "goals_conceded": goals_conceded,

        "goal_difference": goals_scored - goals_conceded,

        "avg_goals_scored": goals_scored / len(last5),

        "avg_goals_conceded": goals_conceded / len(last5)

    }

In [15]:
stats = []

for team in teams["team"]:

    team_stats = compute_team_statistics(team)

    if team_stats is not None:
        stats.append(team_stats)

team_statistics = pd.DataFrame(stats)

print("Number of teams:", len(team_statistics))
print("Missing:", set(teams["team"]) - set(team_statistics["team"]))

Number of teams: 48
Missing: set()


In [16]:
latest_fifa = pd.read_csv("../datasets/fifa_rankings_.csv")
latest_elo = pd.read_csv("../datasets/current_elo_ratings.csv")

In [17]:
latest_fifa = latest_fifa.rename(
    columns={"Team": "team"}
)

In [18]:
print(latest_fifa.columns.tolist())

['team', 'FIFA_Rank']


In [19]:
print(latest_elo.columns.tolist())

['Rank', 'Country_Code', 'Elo_Rating']


In [20]:
import os

print(os.listdir("../datasets"))

['.ipynb_checkpoints', 'current_elo_ratings.csv', 'elo_ratings_wc2026.csv', 'fifa_rankings_.csv', 'historical_elo_ratings.csv', 'results.csv', 'team_statistics.csv', 'wc_2026_fixtures.csv', 'wc_2026_teams.csv']


In [21]:
elo = pd.read_csv("../datasets/elo_ratings_wc2026.csv")

print(elo.columns.tolist())

elo.head()

['year', 'snapshot_date', 'country', 'rank', 'country_code', 'rating', 'rank_max', 'rating_max', 'rank_avg', 'rating_avg', 'rank_min', 'rating_min', 'matches_total', 'matches_home', 'matches_away', 'matches_neutral', 'wins', 'losses', 'draws', 'goals_for', 'goals_against', 'confederation', 'is_host']


,year,snapshot_date,country,rank,country_code,rating,rank_max,rating_max,rank_avg,rating_avg,...,matches_home,matches_away,matches_neutral,wins,losses,draws,goals_for,goals_against,confederation,is_host
0,1901,1901-12-31,England,1,EN,2013,1,2079,2,1989,...,38,35,0,46,16,11,262,102,UEFA,0
1,1901,1901-12-31,Scotland,2,SQ,1973,1,2104,1,2018,...,37,37,0,53,9,12,277,101,UEFA,0
2,1902,1902-12-31,Argentina,1,AR,2021,1,2021,1,2021,...,0,1,0,1,0,0,6,0,CONMEBOL,0
3,1902,1902-12-31,England,2,EN,1995,1,2079,2,1989,...,39,38,0,47,16,14,266,105,UEFA,0
4,1902,1902-12-31,Scotland,3,SQ,1983,1,2104,1,2017,...,39,40,0,56,9,14,293,106,UEFA,0


In [22]:
latest_elo = (
    elo.sort_values("year")
       .groupby("country", as_index=False)
       .last()
)

In [23]:
latest_elo = latest_elo.rename(
    columns={
        "country": "team",
        "rating": "elo_rating"
    }
)

latest_elo = latest_elo[["team", "elo_rating"]]

latest_elo.head()

,team,elo_rating
0,Algeria,1743
1,Argentina,2113
2,Australia,1783
3,Austria,1827
4,Belgium,1867


In [24]:
team_statistics = team_statistics.drop(
    columns=["FIFA_Rank_x", "FIFA_Rank_y"],
    errors="ignore"
)

print(team_statistics.columns.tolist())

['team', 'wins_last5', 'draws_last5', 'losses_last5', 'form_points', 'goals_scored', 'goals_conceded', 'goal_difference', 'avg_goals_scored', 'avg_goals_conceded']


In [25]:
print(team_statistics.columns.tolist())

['team', 'wins_last5', 'draws_last5', 'losses_last5', 'form_points', 'goals_scored', 'goals_conceded', 'goal_difference', 'avg_goals_scored', 'avg_goals_conceded']


In [26]:
print(team_statistics.isnull().sum())

team                  0
wins_last5            0
draws_last5           0
losses_last5          0
form_points           0
goals_scored          0
goals_conceded        0
goal_difference       0
avg_goals_scored      0
avg_goals_conceded    0
dtype: int64


In [27]:
team_statistics = team_statistics.drop(columns=["FIFA_Rank_y"])
team_statistics = team_statistics.rename(
    columns={"FIFA_Rank_x": "FIFA_Rank"}
)

KeyError: "['FIFA_Rank_y'] not found in axis"

In [ ]:
team_statistics = team_statistics.drop(columns=["FIFA_Rank"])

In [ ]:
print(team_statistics.isnull().sum())

In [133]:
latest_fifa["team"] = latest_fifa["team"].replace({
    "Cabo Verde": "Cape Verde",
    "Congo DR": "DR Congo"
})

In [134]:
team_statistics[team_statistics["elo_rating"].isnull()][["team"]]

KeyError: 'elo_rating'

In [135]:
elo["country"] = elo["country"].replace({
    "United States": "USA",
    "Turkey": "Türkiye"
})

In [136]:
latest_elo = (
    elo.sort_values("year")
       .groupby("country", as_index=False)
       .last()
)

latest_elo = latest_elo.rename(
    columns={
        "country": "team",
        "rating": "elo_rating"
    }
)

latest_elo = latest_elo[["team", "elo_rating"]]

In [137]:
team_statistics = team_statistics.drop(columns=["elo_rating"])

team_statistics = team_statistics.merge(
    latest_elo,
    on="team",
    how="left"
)

KeyError: "['elo_rating'] not found in axis"

In [ ]:
team_statistics = team_statistics.merge(
    latest_fifa[["team", "FIFA_Rank"]],
    on="team",
    how="left"
)

In [138]:
team_statistics = team_statistics.merge(
    latest_elo[["team", "elo_rating"]],
    on="team",
    how="left"
)

In [117]:
team_statistics.to_csv(
    "../datasets/team_statistics.csv",
    index=False
)

print("team_statistics.csv saved successfully!")

team_statistics.csv saved successfully!


In [118]:
compute_team_statistics("Brazil")

{'team': 'Brazil',
 'wins_last5': 3,
 'draws_last5': 1,
 'losses_last5': 1,
 'form_points': 10,
 'goals_scored': 13.0,
 'goals_conceded': 7.0,
 'goal_difference': 6.0,
 'avg_goals_scored': 2.6,
 'avg_goals_conceded': 1.4}

In [119]:
compute_team_statistics("USA")

{'team': 'USA',
 'wins_last5': 2,
 'draws_last5': 0,
 'losses_last5': 3,
 'form_points': 6,
 'goals_scored': 10.0,
 'goals_conceded': 12.0,
 'goal_difference': -2.0,
 'avg_goals_scored': 2.0,
 'avg_goals_conceded': 2.4}

In [17]:
compute_team_statistics("Türkiye")

{'team': 'Türkiye',
 'wins_last5': 4,
 'draws_last5': 0,
 'losses_last5': 1,
 'form_points': 12,
 'goals_scored': 8.0,
 'goals_conceded': 3.0,
 'goal_difference': 5.0,
 'avg_goals_scored': 1.6,
 'avg_goals_conceded': 0.6}

In [18]:
compute_team_statistics("Czechia")

{'team': 'Czechia',
 'wins_last5': 2,
 'draws_last5': 2,
 'losses_last5': 1,
 'form_points': 8,
 'goals_scored': 11.0,
 'goals_conceded': 7.0,
 'goal_difference': 4.0,
 'avg_goals_scored': 2.2,
 'avg_goals_conceded': 1.4}

In [19]:
team_statistics = []

for team in teams_list:

    stats = compute_team_statistics(team)

    if stats is not None:

        team_statistics.append(stats)

team_statistics = pd.DataFrame(team_statistics)

team_statistics.head()

,team,wins_last5,draws_last5,losses_last5,form_points,goals_scored,goals_conceded,goal_difference,avg_goals_scored,avg_goals_conceded
0,Algeria,3,1,1,10,12.0,3.0,9.0,2.4,0.6
1,Argentina,5,0,0,15,15.0,1.0,14.0,3.0,0.2
2,Australia,3,1,1,10,9.0,3.0,6.0,1.8,0.6
3,Austria,4,1,0,13,11.0,3.0,8.0,2.2,0.6
4,Belgium,3,2,0,11,14.0,4.0,10.0,2.8,0.8


In [20]:
print(team_statistics.shape)

(48, 10)


In [21]:
print(team_statistics["team"].nunique())

48


In [22]:
print(set(teams["team"]) - set(team_statistics["team"]))

set()


In [23]:
print(set(teams["team"]) - set(team_statistics["team"]))

set()


In [24]:
latest_fifa = fifa_rankings.rename(
    columns={
        "Team": "team",
        "FIFA Rank": "FIFA_Rank"
    }
)

latest_fifa.head()

,team,FIFA_Rank
0,Argentina,1
1,Spain,2
2,France,3
3,England,4
4,Portugal,5


In [25]:
team_statistics = team_statistics.merge(

    latest_fifa,

    on="team",

    how="left"

)

team_statistics.head()

,team,wins_last5,draws_last5,losses_last5,form_points,goals_scored,goals_conceded,goal_difference,avg_goals_scored,avg_goals_conceded,FIFA_Rank
0,Algeria,3,1,1,10,12.0,3.0,9.0,2.4,0.6,28.0
1,Argentina,5,0,0,15,15.0,1.0,14.0,3.0,0.2,1.0
2,Australia,3,1,1,10,9.0,3.0,6.0,1.8,0.6,27.0
3,Austria,4,1,0,13,11.0,3.0,8.0,2.2,0.6,24.0
4,Belgium,3,2,0,11,14.0,4.0,10.0,2.8,0.8,9.0


In [26]:
print(team_statistics.isnull().sum())

team                  0
wins_last5            0
draws_last5           0
losses_last5          0
form_points           0
goals_scored          0
goals_conceded        0
goal_difference       0
avg_goals_scored      0
avg_goals_conceded    0
FIFA_Rank             3
dtype: int64


In [27]:
team_statistics[
    team_statistics["FIFA_Rank"].isnull()
][["team"]]

,team
8,Cape Verde
13,DR Congo
29,New Zealand


In [28]:
latest_fifa[
    latest_fifa["team"].str.contains(
        "Cape|Congo|Zealand",
        case=False,
        na=False
    )
]

,team,FIFA_Rank
45,Congo DR,46


In [29]:
latest_fifa["team"] = latest_fifa["team"].replace({

    "Cabo Verde": "Cape Verde",

    "Congo DR": "DR Congo",

    "New Zealand": "New Zealand"

})

In [30]:
team_statistics = team_statistics.drop(
    columns=["FIFA_Rank"]
)

team_statistics = team_statistics.merge(

    latest_fifa,

    on="team",

    how="left"

)

team_statistics.isnull().sum()

team                  0
wins_last5            0
draws_last5           0
losses_last5          0
form_points           0
goals_scored          0
goals_conceded        0
goal_difference       0
avg_goals_scored      0
avg_goals_conceded    0
FIFA_Rank             1
dtype: int64

In [31]:
team_statistics[
    team_statistics["FIFA_Rank"].isnull()
][["team"]]

,team
29,New Zealand


In [32]:
latest_fifa[
    latest_fifa["team"].str.contains(
        "Zealand",
        case=False,
        na=False
    )
]

,team,FIFA_Rank


In [33]:
latest_fifa[
    latest_fifa["team"].str.contains(
        "New",
        case=False,
        na=False
    )
]

,team,FIFA_Rank
183,New Caledonia,184
186,Papua New Guinea,187


In [34]:
sorted(latest_fifa["team"].unique())

['Albania',
 'Algeria',
 'American Samoa',
 'Andorra',
 'Angola',
 'Anguilla',
 'Antigua and Barbuda',
 'Argentina',
 'Armenia',
 'Aruba',
 'Australia',
 'Austria',
 'Azerbaijan',
 'Bahamas',
 'Bahrain',
 'Bangladesh',
 'Barbados',
 'Belarus',
 'Belgium',
 'Belize',
 'Benin',
 'Bermuda',
 'Bhutan',
 'Bosnia and Herzegovina',
 'Botswana',
 'Brazil',
 'British Virgin Islands',
 'Brunei Darussalam',
 'Bulgaria',
 'Burkina Faso',
 'Burundi',
 'Cambodia',
 'Cameroon',
 'Canada',
 'Cape Verde',
 'Cayman Islands',
 'Central African Republic',
 'Chad',
 'China',
 'Chinese Taipei',
 'Colombia',
 'Comoros',
 'Cook Islands',
 'Costa Rica',
 'Croatia',
 'Cuba',
 'Curaçao',
 'Cyprus',
 'Czechia',
 'DR Congo',
 'Denmark',
 'Djibouti',
 'Dominica',
 'Dominican Republic',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'England',
 'Equatorial Guinea',
 'Eritrea',
 'Estonia',
 'Eswatini',
 'Ethiopia',
 'Faroe Islands',
 'Fiji',
 'Finland',
 'France',
 'Gabon',
 'Gambia',
 'Georgia',
 'Germany',
 'Ghana',
 'Gibr

In [35]:
latest_fifa["team"] = latest_fifa["team"].replace({
    "Exact Name From FIFA": "New Zealand"
})

In [36]:
team_statistics = team_statistics.drop(
    columns=["FIFA_Rank"]
)

team_statistics = team_statistics.merge(
    latest_fifa,
    on="team",
    how="left"
)

team_statistics.isnull().sum()

team                  0
wins_last5            0
draws_last5           0
losses_last5          0
form_points           0
goals_scored          0
goals_conceded        0
goal_difference       0
avg_goals_scored      0
avg_goals_conceded    0
FIFA_Rank             1
dtype: int64

In [37]:
team_statistics.loc[
    team_statistics["team"] == "New Zealand",
    "FIFA_Rank"
] = 86

In [38]:
team_statistics.isnull().sum()

team                  0
wins_last5            0
draws_last5           0
losses_last5          0
form_points           0
goals_scored          0
goals_conceded        0
goal_difference       0
avg_goals_scored      0
avg_goals_conceded    0
FIFA_Rank             0
dtype: int64

In [39]:
team_statistics[
    team_statistics["team"] == "New Zealand"
]

,team,wins_last5,draws_last5,losses_last5,form_points,goals_scored,goals_conceded,goal_difference,avg_goals_scored,avg_goals_conceded,FIFA_Rank
29,New Zealand,1,1,3,4,6.0,10.0,-4.0,1.2,2.0,86.0


In [44]:
latest_elo = elo_ratings.sort_values("year").drop_duplicates(
    subset="country",
    keep="last"
)

In [45]:
latest_elo = latest_elo.rename(
    columns={
        "country": "team",
        "rating": "elo_rating"
    }
)

latest_elo = latest_elo[
    ["team", "elo_rating"]
]

latest_elo.head()

,team,elo_rating
4598,Norway,1912
4632,Ghana,1503
4606,Paraguay,1833
4605,Mexico,1860
4604,Belgium,1867


In [46]:
globals().keys()

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__session__', 'json', 'getpass', 'hashlib', 'import_pandas_safely', '__pandas', 'is_data_frame', 'dataframe_columns', 'dtypes_str', 'dataframe_hash', 'get_dataframes', '_1', '_i', '_ii', '_iii', '_i1', 'pd', 'np', 'joblib', '_i2', 'matches', 'fifa_rankings', 'elo_ratings', 'fixtures', 'teams', '_i3', 'model', 'feature_columns', '_i4', 'recent_matches', '_i5', 'completed_matches', '_i6', '_i7', 'teams_list', '_8', '_i8', 'TEAM_NAME_MAP', '_i9', 'last_five_matches', '_i10', '_10', '_i11', '_11', '_i12', '_12', '_i13', '_13', '_i14', 'compute_team_statistics', '_i15', '_15', '_i16', '_16', '_i17', '_17', '_i18', '_18', '_i19', 'team_statistics', 'team', 'stats', '_19', '_i20', '_i21', '_i22', '_i23', '_i24', 'latest_fifa', '_24', '_i25', '_25', '_i26', '_i27', '_27', '_i28', '_28', '_i29', '_i3

In [47]:
print(latest_elo.columns)

Index(['team', 'elo_rating'], dtype='object')


In [48]:
set(team_statistics["team"]) - set(latest_elo["team"])

{'Türkiye', 'USA'}

In [52]:
latest_elo["team"] = latest_elo["team"].replace({
    "Turkey": "Türkiye",
    "United States": "USA"
})

In [53]:
team_statistics = team_statistics.merge(

    latest_elo,

    on="team",

    how="left"

)

In [56]:
team_statistics = team_statistics.drop(columns=["elo_rating_x"])

team_statistics = team_statistics.rename(
    columns={
        "elo_rating_y": "elo_rating"
    }
)

team_statistics.isnull().sum()


team                  0
wins_last5            0
draws_last5           0
losses_last5          0
form_points           0
goals_scored          0
goals_conceded        0
goal_difference       0
avg_goals_scored      0
avg_goals_conceded    0
FIFA_Rank             0
elo_rating            0
dtype: int64

In [57]:
team_statistics.isnull().sum()

team                  0
wins_last5            0
draws_last5           0
losses_last5          0
form_points           0
goals_scored          0
goals_conceded        0
goal_difference       0
avg_goals_scored      0
avg_goals_conceded    0
FIFA_Rank             0
elo_rating            0
dtype: int64

In [58]:
set(team_statistics["team"]) - set(latest_elo["team"])

set()

# Creating Match Features

The trained XGBoost model expects a fixed set of engineered features for every match.

This section dynamically constructs those features from the statistics of the selected home and away teams.

In [59]:
def get_team(team):

    row = team_statistics[
        team_statistics["team"] == team
    ]

    if row.empty:
        raise ValueError(f"{team} not found.")

    return row.iloc[0]
    

In [60]:
get_team("Brazil")

team                  Brazil
wins_last5                 3
draws_last5                1
losses_last5               1
form_points               10
goals_scored            13.0
goals_conceded           7.0
goal_difference          6.0
avg_goals_scored         2.6
avg_goals_conceded       1.4
FIFA_Rank                6.0
elo_rating              1984
Name: 6, dtype: object

In [61]:
def create_match_features(

    home_team,

    away_team,

    tournament_type=0,

    tournament_importance=5,

    host_advantage=0

):

    home = get_team(home_team)

    away = get_team(away_team)

    features = {

        "home_elo": home["elo_rating"],

        "away_elo": away["elo_rating"],

        "elo_difference":
        home["elo_rating"] - away["elo_rating"],


        "home_fifa_rank": home["FIFA_Rank"],

        "away_fifa_rank": away["FIFA_Rank"],

        "rank_difference":
        away["FIFA_Rank"] - home["FIFA_Rank"],


        "home_form_points":
        home["form_points"],

        "away_form_points":
        away["form_points"],

        "form_difference":
        home["form_points"] - away["form_points"],


        "home_goal_diff":
        home["goal_difference"],

        "away_goal_diff":
        away["goal_difference"],

        "goal_diff_difference":
        home["goal_difference"] - away["goal_difference"],


        "home_wins_last5":
        home["wins_last5"],

        "away_wins_last5":
        away["wins_last5"],

        "wins_difference":
        home["wins_last5"] - away["wins_last5"],


        "host_advantage":
        host_advantage,

        "tournament_type":
        tournament_type,

        "tournament_importance":
        tournament_importance

    }

    features = pd.DataFrame([features])

    features = features[feature_columns]

    return features

In [62]:
features = create_match_features(

    "Brazil",

    "Argentina"

)

features

,home_elo,away_elo,elo_difference,home_fifa_rank,away_fifa_rank,rank_difference,home_form_points,away_form_points,form_difference,home_goal_diff,away_goal_diff,goal_diff_difference,home_wins_last5,away_wins_last5,wins_difference,host_advantage,tournament_type,tournament_importance
0,1984,2113,-129,6.0,1.0,-5.0,10,15,-5,6.0,14.0,-8.0,3,5,-2,0,0,5


# Predicting Match Outcome

This function generates the required feature vector for two selected teams and uses the trained XGBoost model to predict the probabilities of Home Win, Draw, and Away Win.

In [72]:
def predict_match(
    home_team,
    away_team,
    tournament="FIFA World Cup",
    host_team=None
):

    importance = {
        "Friendly": 1,
        "Nations League": 2,
        "Qualifier": 3,
        "Continental Cup": 4,
        "FIFA World Cup": 5
    }

    tournament_importance = importance.get(tournament, 1)

    tournament_type = 0

    host_advantage = 0

    if host_team is not None:
        if home_team == host_team:
            host_advantage = 1

    features = create_match_features(
        home_team,
        away_team,
        tournament_type,
        tournament_importance,
        host_advantage
    )

    features = features[feature_columns]

    probabilities = model.predict_proba(features)[0]

    winner = np.argmax(probabilities)

    if winner == 0:
        prediction = f"{away_team} Win"
    elif winner == 1:
        prediction = "Draw"
    else:
        prediction = f"{home_team} Win"

    print("=" * 45)
    print(f"{home_team} vs {away_team}")
    print("=" * 45)
    print()

    print(f"{away_team} Win : {probabilities[0]:.2%}")
    print(f"Draw            : {probabilities[1]:.2%}")
    print(f"{home_team} Win : {probabilities[2]:.2%}")
    print()

    print(f"Prediction : {prediction}")

    return probabilities

In [73]:
predict_match(

    "Brazil",

    "Argentina"

)

Brazil vs Argentina

Argentina Win : 58.59%
Draw            : 26.93%
Brazil Win : 14.48%

Prediction : Argentina Win


array([0.585866  , 0.26931798, 0.14481606], dtype=float32)

In [74]:
predict_match(

    "USA",

    "England",

    host_team="USA"

)

USA vs England

England Win : 79.40%
Draw            : 15.11%
USA Win : 5.48%

Prediction : England Win


array([0.7940366 , 0.15114284, 0.05482051], dtype=float32)

In [75]:
predict_match(

    "Spain",

    "Germany"

)

Spain vs Germany

Germany Win : 6.50%
Draw            : 24.82%
Spain Win : 68.68%

Prediction : Spain Win


array([0.06503343, 0.2481782 , 0.6867884 ], dtype=float32)

In [76]:
features = create_match_features(
    "USA",
    "England",
    tournament_type=0,
    tournament_importance=5,
    host_advantage=1
)

features

,home_elo,away_elo,elo_difference,home_fifa_rank,away_fifa_rank,rank_difference,home_form_points,away_form_points,form_difference,home_goal_diff,away_goal_diff,goal_diff_difference,home_wins_last5,away_wins_last5,wins_difference,host_advantage,tournament_type,tournament_importance
0,1721,2020,-299,17.0,4.0,-13.0,6,10,-4,-2.0,5.0,-7.0,2,3,-1,1,0,5


In [77]:
get_team("USA")

team                   USA
wins_last5               2
draws_last5              0
losses_last5             3
form_points              6
goals_scored          10.0
goals_conceded        12.0
goal_difference       -2.0
avg_goals_scored       2.0
avg_goals_conceded     2.4
FIFA_Rank             17.0
elo_rating            1721
Name: 45, dtype: object

In [78]:
get_team("England")

team                  England
wins_last5                  3
draws_last5                 1
losses_last5                1
form_points                10
goals_scored              9.0
goals_conceded            4.0
goal_difference           5.0
avg_goals_scored          1.8
avg_goals_conceded        0.8
FIFA_Rank                 4.0
elo_rating               2020
Name: 16, dtype: object

In [79]:
features = create_match_features(
    "USA",
    "England",
    tournament_type=0,
    tournament_importance=5,
    host_advantage=1
)

features.T

,0
home_elo,1721.0
away_elo,2020.0
elo_difference,-299.0
home_fifa_rank,17.0
away_fifa_rank,4.0
rank_difference,-13.0
home_form_points,6.0
away_form_points,10.0
form_difference,-4.0
home_goal_diff,-2.0


In [84]:
print(team_statistics.columns.tolist())

['team', 'wins_last5', 'draws_last5', 'losses_last5', 'form_points', 'goals_scored', 'goals_conceded', 'goal_difference', 'avg_goals_scored', 'avg_goals_conceded']


In [85]:
team_statistics.head()

,team,wins_last5,draws_last5,losses_last5,form_points,goals_scored,goals_conceded,goal_difference,avg_goals_scored,avg_goals_conceded
0,Mexico,5,0,0,15,11.0,1.0,10.0,2.2,0.2
1,South Africa,1,2,2,5,3.0,5.0,-2.0,0.6,1.0
2,South Korea,3,0,2,9,8.0,3.0,5.0,1.6,0.6
3,Czechia,2,2,1,8,11.0,7.0,4.0,2.2,1.4
4,Canada,2,3,0,9,10.0,2.0,8.0,2.0,0.4


In [28]:
team_statistics = team_statistics.merge(
    latest_fifa[["team", "FIFA_Rank"]],
    on="team",
    how="left"
)

team_statistics = team_statistics.merge(
    latest_elo,
    on="team",
    how="left"
)

In [29]:
print(team_statistics.columns.tolist())
print()

print(team_statistics.isnull().sum())

['team', 'wins_last5', 'draws_last5', 'losses_last5', 'form_points', 'goals_scored', 'goals_conceded', 'goal_difference', 'avg_goals_scored', 'avg_goals_conceded', 'FIFA_Rank', 'elo_rating']

team                  0
wins_last5            0
draws_last5           0
losses_last5          0
form_points           0
goals_scored          0
goals_conceded        0
goal_difference       0
avg_goals_scored      0
avg_goals_conceded    0
FIFA_Rank             3
elo_rating            2
dtype: int64


In [30]:
team_statistics = team_statistics.drop(
    columns=[
        "FIFA_Rank_y",
        "elo_rating_y",
        "FIFA_Rank"
    ],
    errors="ignore"
)

team_statistics = team_statistics.rename(
    columns={
        "FIFA_Rank_x": "FIFA_Rank",
        "elo_rating_x": "elo_rating"
    }
)

In [31]:
print(team_statistics.columns)

print()

print(team_statistics.isnull().sum())

Index(['team', 'wins_last5', 'draws_last5', 'losses_last5', 'form_points',
       'goals_scored', 'goals_conceded', 'goal_difference', 'avg_goals_scored',
       'avg_goals_conceded', 'elo_rating'],
      dtype='object')

team                  0
wins_last5            0
draws_last5           0
losses_last5          0
form_points           0
goals_scored          0
goals_conceded        0
goal_difference       0
avg_goals_scored      0
avg_goals_conceded    0
elo_rating            2
dtype: int64


In [32]:
print(team_statistics.columns.tolist())

['team', 'wins_last5', 'draws_last5', 'losses_last5', 'form_points', 'goals_scored', 'goals_conceded', 'goal_difference', 'avg_goals_scored', 'avg_goals_conceded', 'elo_rating']


In [33]:

print(team_statistics.isnull().sum())

team                  0
wins_last5            0
draws_last5           0
losses_last5          0
form_points           0
goals_scored          0
goals_conceded        0
goal_difference       0
avg_goals_scored      0
avg_goals_conceded    0
elo_rating            2
dtype: int64


In [34]:
team_statistics = team_statistics.merge(
    latest_fifa[["team", "FIFA_Rank"]],
    on="team",
    how="left"
)

team_statistics = team_statistics.merge(
    latest_elo,
    on="team",
    how="left"
)

In [35]:
print(team_statistics.columns.tolist())
print()

print(team_statistics.isnull().sum())

['team', 'wins_last5', 'draws_last5', 'losses_last5', 'form_points', 'goals_scored', 'goals_conceded', 'goal_difference', 'avg_goals_scored', 'avg_goals_conceded', 'elo_rating_x', 'FIFA_Rank', 'elo_rating_y']

team                  0
wins_last5            0
draws_last5           0
losses_last5          0
form_points           0
goals_scored          0
goals_conceded        0
goal_difference       0
avg_goals_scored      0
avg_goals_conceded    0
elo_rating_x          2
FIFA_Rank             3
elo_rating_y          2
dtype: int64
